# EDA

A quick, visual look at each product's snapshot before modelling: size, missingness, target prevalence, feature distributions, and which features move with the target. Nothing is dropped here — that happens in Feature Checks. Plotting uses the small reusable helpers in `src/plots.py` so every product section looks the same.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from configs._schema import load_config
cfg = load_config(ROOT / 'configs/fx_activation.yaml')   # validates on load
cfg.product, str(cfg.obs_date), len(cfg.features)

from src.dataset import load_labelled
from src import plots
import matplotlib.pyplot as plt

X, y = load_labelled(spark, cfg, cfg.obs_date)   # features + materialised target
X.shape, round(float(y.mean()), 4)

## 1. FX Activation

### Size, dtypes, missingness

In [ ]:
print(X[cfg.features].dtypes.value_counts())
plots.plot_missingness(X, cfg.features, top=15)
plt.show()

### Target prevalence
The share of the eligible population that activates — also the baseline every model is measured against.

In [ ]:
print('base rate:', round(float(y.mean()), 4))
y.value_counts().plot.bar(title='target counts')
plt.show()

### Feature distributions
Histogram of a feature, split by the target so you can see whether the two groups separate.

In [ ]:
plots.plot_hist(X['avg_balance_3m'], by=y, bins=40)
plt.show()

### Feature vs target rate
Bin a numeric feature and plot the target rate per bin; the dashed line is the overall base rate. A monotonic climb suggests the feature carries real signal.

In [ ]:
plots.plot_target_rate(X['tenure_months'], y, bins=10)
plt.show()

### Correlations among numeric features

In [ ]:
num = X[cfg.features].select_dtypes('number').columns.tolist()[:15]
plots.plot_corr(X, num)
plt.show()

### Carry-forward notes
Jot concerns (suspicious or weak features). They are inputs to the human decision in Feature Checks — no drops are made here.